In [1]:
# Robustness Experiments

# This notebook evaluates whether the main cross-domain findings remain under alternative data controls.
# Three analyses are considered: operating-system-only data, temporally matched training data,
# and strict leakage-group deduplication.

In [2]:
import re

import pandas as pd

from src.config import (
    TEXT_COL,
    LABEL_COL,
    GROUP_COL,
    ID_COL,
    SOFTWARE_COL,
    DATE_COL,
    CWE_LABELS,
    RESULTS_ROBUSTNESS_DIR,
)

from src.data import (
    load_preprocessed_data
)

from src.models import (
    fit_predict_classical
)

from src.evaluation import (
    evaluate_predictions
)


ru_df, nonru_df = (
    load_preprocessed_data()
)


ru_dev = (
    ru_df[
        ru_df["split"]
        .isin(["train", "val"])
    ]
    .copy()
)

ru_test = (
    ru_df[
        ru_df["split"] == "test"
    ]
    .copy()
)

nonru_dev = (
    nonru_df[
        nonru_df["split"]
        .isin(["train", "val"])
    ]
    .copy()
)


MODELS = {
    "Logistic Regression":
        "word",

    "Linear SVM":
        "char",
}

In [3]:
## Define the Common Robustness Evaluation

# A shared evaluation function is defined so that each robustness dataset is tested
# with the same selected Logistic Regression and Linear SVM configurations and the same evaluation metrics.

In [4]:
def run_robustness(
    train_df,
    test_df,
    labels,
    name
):

    rows = []

    for model_name, representation in (
        MODELS.items()
    ):

        pred, n_features, runtime = (
            fit_predict_classical(
                train_df,
                test_df,
                model_name,
                representation,
                TEXT_COL,
                LABEL_COL
            )
        )

        metrics, _ = (
            evaluate_predictions(
                test_df[LABEL_COL],
                pred,
                labels
            )
        )

        rows.append({
            "robustness":
                name,

            "model":
                model_name,

            "train_records":
                len(train_df),

            "test_records":
                len(test_df),

            **metrics,

            "n_features":
                n_features,

            "runtime_seconds":
                runtime,
        })

    return rows

In [5]:
## Construct the Operating-System-Only Subsets

# Records associated with operating systems are isolated in both domains
# and the Russian test set. Only CWE classes represented in all required
# subsets are retained to support a consistent comparison.

In [6]:
def is_os_record(value):

    if pd.isna(value):
        return False

    return bool(
        re.search(
            r"операционн|operating\s*system",
            str(value).lower()
        )
    )


ru_dev_os = (
    ru_dev[
        ru_dev[SOFTWARE_COL]
        .apply(is_os_record)
    ]
    .copy()
)

nonru_dev_os = (
    nonru_dev[
        nonru_dev[SOFTWARE_COL]
        .apply(is_os_record)
    ]
    .copy()
)

ru_test_os = (
    ru_test[
        ru_test[SOFTWARE_COL]
        .apply(is_os_record)
    ]
    .copy()
)


OS_LABELS = sorted(
    set(ru_dev_os[LABEL_COL])
    &
    set(nonru_dev_os[LABEL_COL])
    &
    set(ru_test_os[LABEL_COL])
)


ru_dev_os = (
    ru_dev_os[
        ru_dev_os[LABEL_COL]
        .isin(OS_LABELS)
    ]
)

nonru_dev_os = (
    nonru_dev_os[
        nonru_dev_os[LABEL_COL]
        .isin(OS_LABELS)
    ]
)

ru_test_os = (
    ru_test_os[
        ru_test_os[LABEL_COL]
        .isin(OS_LABELS)
    ]
)


print(
    len(ru_dev_os),
    len(nonru_dev_os),
    len(ru_test_os),
    len(OS_LABELS)
)

11663 16378 2069 30


In [7]:
## Evaluate Operating-System-Only Transfer

# Russian-trained and non-Russian-trained classical models are evaluated
# on the operating-system-only Russian test subset. This controls for
# the strong difference in software-type composition between the two domains.

In [8]:
os_results = []

for domain, train_df in {
    "RU":
        ru_dev_os,

    "Non-RU":
        nonru_dev_os,
}.items():

    rows = run_robustness(
        train_df,
        ru_test_os,
        OS_LABELS,
        "OS-only"
    )

    for row in rows:
        row["training_domain"] = domain

    os_results.extend(rows)


os_results_df = (
    pd.DataFrame(os_results)
)

In [9]:
## Define Temporal Matching

# Publication years are extracted and a sampling procedure is defined that matches
# the non-Russian development data to the year distribution of the Russian development set.

In [10]:
for dataframe in [
    ru_dev,
    nonru_dev,
    ru_test
]:

    dataframe["year"] = (
        pd.to_datetime(
            dataframe[DATE_COL],
            errors="coerce",
            dayfirst=True
        )
        .dt.year
    )


TEMPORAL_SEEDS = [
    111, 222, 333, 444, 555,
    666, 777, 888, 999, 1111
]


def temporal_match(
    ru_reference,
    nonru_pool,
    seed
):

    parts = []

    for year, count in (
        ru_reference["year"]
        .value_counts()
        .items()
    ):

        pool = (
            nonru_pool[
                nonru_pool["year"]
                == year
            ]
        )

        if len(pool) < count:
            raise ValueError(
                f"Not enough records for {year}"
            )

        parts.append(
            pool.sample(
                n=count,
                random_state=(
                    seed + int(year)
                )
            )
        )


    return (
        pd.concat(
            parts,
            ignore_index=True
        )
        .sample(
            frac=1,
            random_state=seed
        )
        .reset_index(drop=True)
    )

In [11]:
## Evaluate Temporally Matched Training Data

# Ten temporally matched non-Russian samples are evaluated against
# the fixed Russian test set. Mean and standard deviation of Macro-F1
# are calculated across the repeated samples.

In [12]:
temporal_results = []


for seed in TEMPORAL_SEEDS:

    matched = temporal_match(
        ru_dev,
        nonru_dev,
        seed
    )

    assert not (
        set(matched[GROUP_COL])
        &
        set(ru_test[GROUP_COL])
    )

    rows = run_robustness(
        matched,
        ru_test,
        CWE_LABELS,
        "Temporal matched"
    )

    for row in rows:
        row["seed"] = seed

    temporal_results.extend(rows)


temporal_results_df = (
    pd.DataFrame(
        temporal_results
    )
)


temporal_summary = (
    temporal_results_df
    .groupby("model")
    .agg(
        macro_f1_mean=(
            "macro_f1",
            "mean"
        ),
        macro_f1_sd=(
            "macro_f1",
            "std"
        )
    )
    .reset_index()
)

In [13]:
## Construct Strictly Deduplicated Datasets

# Leakage groups containing conflicting CWE labels are removed and one representative
# record is retained from each remaining group. The resulting datasets provide a stricter test
# of whether repeated related records influence performance.

In [14]:
def strict_deduplicate(df):

    valid = (
        df[
            ~df["ambiguous_cwe_group"]
            .fillna(False)
        ]
        .copy()
    )

    return (
        valid
        .sort_values(ID_COL)
        .drop_duplicates(
            GROUP_COL
        )
        .reset_index(drop=True)
    )


ru_dev_dedup = (
    strict_deduplicate(
        ru_dev
    )
)

nonru_dev_dedup = (
    strict_deduplicate(
        nonru_dev
    )
)

ru_test_dedup = (
    strict_deduplicate(
        ru_test
    )
)


DEDUP_LABELS = sorted(
    set(ru_dev_dedup[LABEL_COL])
    &
    set(nonru_dev_dedup[LABEL_COL])
    &
    set(ru_test_dedup[LABEL_COL])
)

In [15]:
## Evaluate Strictly Deduplicated Data

# The deduplicated datasets are restricted to their shared CWE label space and the Russian-
# and non-Russian-trained classical models are evaluated on the deduplicated Russian test subset.

In [16]:
ru_dev_dedup = (
    ru_dev_dedup[
        ru_dev_dedup[LABEL_COL]
        .isin(DEDUP_LABELS)
    ]
)

nonru_dev_dedup = (
    nonru_dev_dedup[
        nonru_dev_dedup[LABEL_COL]
        .isin(DEDUP_LABELS)
    ]
)

ru_test_dedup = (
    ru_test_dedup[
        ru_test_dedup[LABEL_COL]
        .isin(DEDUP_LABELS)
    ]
)


dedup_results = []


for domain, train_df in {
    "RU":
        ru_dev_dedup,

    "Non-RU":
        nonru_dev_dedup,
}.items():

    rows = run_robustness(
        train_df,
        ru_test_dedup,
        DEDUP_LABELS,
        "Strict dedup"
    )

    for row in rows:
        row["training_domain"] = domain

    dedup_results.extend(rows)


dedup_results_df = (
    pd.DataFrame(
        dedup_results
    )
)

In [17]:
## Save Robustness Results

# Results from the operating-system-only, temporal-matching, and strict-deduplication experiments
# are exported for comparison with the main cross-domain findings.

In [18]:
os_results_df.to_csv(
    RESULTS_ROBUSTNESS_DIR
    / "OS_only_results.csv",
    index=False
)

temporal_results_df.to_csv(
    RESULTS_ROBUSTNESS_DIR
    / "temporal_all_runs.csv",
    index=False
)

temporal_summary.to_csv(
    RESULTS_ROBUSTNESS_DIR
    / "temporal_summary.csv",
    index=False
)

dedup_results_df.to_csv(
    RESULTS_ROBUSTNESS_DIR
    / "strict_dedup_results.csv",
    index=False
)